# Lahore AQI -- Exploratory Data Analysis

This notebook explores the aggregated hourly AQI feature data for Lahore,
pulled from the Feast offline feature store. The goal is to identify
temporal trends, understand feature distributions and relationships, and
surface data-quality characteristics that explain modeling decisions made
elsewhere in the pipeline (e.g. the chronological train/val/test split,
the `aqi_change_rate` staleness threshold).

All plotting logic lives in `eda_plots.py` this notebook only calls
those functions and explains those plots.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

import os
os.chdir("C:\\Users\\abuhu\\Desktop\\Internship\\AQI-Pearls-Predictor")
print(os.getcwd())

from feast_store import get_all_rows
import eda_plots as eda
from clean_backfill import clean

pd.set_option("display.max_columns", None)


## 1. Load the data

Pulling the full historical feature set from Feast (`get_all_rows()`),
which returns the same shape used by the training pipeline: `timestamp`
as epoch-ms int, plus `aqi, pm25, pm10, temperature, humidity,
aqi_change_rate, hour, day_of_week, month`.


In [ ]:
df = get_all_rows()
print(f"{len(df)} raw rows loaded")

df = clean(df)
print(f"{len(df)} rows after cleaning (dedup + change-rate recompute)")

df.head()


In [ ]:
df.describe(include="all")


## 2. Temporal patterns

### 2.1 Full AQI history

The background shading marks EPA AQI hazard categories (Good -> Hazardous),
so it's easy to see at a glance how much of Lahore's AQI history sits in
"Unhealthy" territory or worse.


In [ ]:
fig = eda.plot_aqi_timeseries(df)
plt.show()


### 2.2 Diurnal pattern (hour of day)

Air quality in cities driven by traffic and industrial activity often
shows a bimodal pattern meaning: worse during morning/evening rush hours, better
overnight and midday.


In [ ]:
fig = eda.plot_hourly_pattern(df)
plt.show()


### 2.3 Day-of-week pattern

In [ ]:
fig = eda.plot_weekly_pattern(df)
plt.show()


### 2.4 Monthly pattern

Currently meaningless since I do not have the necessary amount of data (spanning of multiple months; 2 at the least)


In [ ]:
fig = eda.plot_monthly_pattern(df)
plt.show()


## 3. Distributions

### 3.1 AQI distribution

Skewness matters here: AQI is bounded below at 0 and can spike sharply
upward during pollution events, so a right-skewed distribution is
expected.


In [ ]:
fig = eda.plot_aqi_distribution(df)
plt.show()


### 3.2 Pollutant distributions (PM2.5 / PM10)

In [ ]:
fig = eda.plot_pollutant_distributions(df)
plt.show()


## 4. Relationships between features

### 4.1 Correlation heatmap

This map shows which features are strongly correlated and which are not.


In [ ]:
fig = eda.plot_correlation_heatmap(df)
plt.show()


### 4.2 AQI vs. PM2.5 / PM10

Since AQI is computed from pollutant concentrations, a clear positive
relationship here is expected.


In [ ]:
fig = eda.plot_aqi_vs_pm(df)
plt.show()


### 4.3 AQI change rate distribution

This is the raw Feast feature (nulls included, not the zero-imputed
version `train.py` passes to the models).


In [ ]:
fig = eda.plot_change_rate_distribution(df)
plt.show()


## 5. Data quality

### 5.1 Missingness / gaps over time

Expected gap between consecutive rows is 1 hour (hourly fetch cadence).
Spikes above the dashed reference line indicate missed fetch cycles,
stale-data exclusions during validation, or GitHub Actions runner issues.


In [ ]:
fig = eda.plot_missingness_over_time(df)
plt.show()


### 5.2 Train / validation / test split variance


In [ ]:
fig = eda.plot_train_val_test_variance(df)
plt.show()


## 6. Summary of findings

### Row count / history span:
Out of 196 rows only 180 were unique hourly readings

On average the AQI of Lahore is 114 which is already unhealthy, mostly for sensitive people. With a std of 25 which means at mostly stays in the moderate to unhealthy bands
At max according to last few days the max it has reached is 186 which unhealthy even for non sensitive people. Any more and it would be hazardous to stay in the city
### Diurnal / weekly / monthly patterns observed:
Over the course of the day, you can see a noticeable dip in aqi during the mid day which is expected due to sunlight and wind and a rise during the night due to increase traffic in the afternoon and reduced boundary layer height which traps the emissions near the surface

You see an increase aqi trend on Thursdays and Fridays and lower Mondays and Sundays probably due to decrease traffic activity

The AQI distribution is right skewed because of the bad but still non extremely hazardous levels of aqi in Lahore. AQI values tend to stay closer to 0 which is the normal behaviour
### Distribution shape notes (skewness, outliers):
PM2.5 which is primarily the pollutant in Lahore has similar distribution to aqi(expected)
PM10 is right skewed which is the expected distribution.
### Correlation highlights (strongest AQI drivers):
Heavy correlation between pm2.5, pm10 and aqi values. Expected because the pollutant values are related and show pollutants in a city and aqi is simply the maximum of those values. After that AQI is heavily related to humidity and after that temperature is also related to aqi
### Data quality issues found (gaps, stale periods):
Some gaps in the data due to GitHub actions and hopsworks materialization problems and also a big gap at the end while switching with feast
### How train/val/test variance explains the R² behavior:
High variation in the train part. Slightly lower variation in the validation and test splits. Primarily because of the gap filled by openaq. It was necessary tho. But it explains the bad R²